In [10]:
import pandas as pd
from pandas_ta.momentum import rsi
from tqdm.notebook import tqdm

import redis.asyncio as redis  
import asyncio
from datetime import datetime
import nest_asyncio

import os
from alpaca.trading.client import TradingClient
from alpaca.trading.requests import MarketOrderRequest
from alpaca.trading.enums import OrderSide

API_KEY = os.environ['API_KEY']
SECRET_KEY = os.environ['SECRET_KEY']

nest_asyncio.apply()
redis_client = redis.Redis(host='localhost', port=6379, decode_responses=True)
trading_client = TradingClient(API_KEY,SECRET_KEY, paper=True)

In [19]:
tickers = ['1','2','3']

In [20]:
async def place_market_order(ticker, side):
    pass

In [41]:
async def consume_ohlc_data(ticker, last_n=15):
    """
    Fetch last N ticks, then listen for new OHLC data in real-time. Awaits strategy defined in strategy config
    """

    stream_key = f"test_{ticker}"
    
    messages = await redis_client.xrevrange(stream_key, count=last_n)
    messages.reverse() 
    hist_data = []
    
    progress_bar = tqdm(desc=f"{ticker}", bar_format="{n} {l_bar} {postfix}") 
    
    for entry_id, data in messages:
        data['close'] = float(data['close'])
        data['timestamp'] = datetime.fromisoformat(data['timestamp'])
        
        hist_data.append(data)
        last_id = entry_id
        progress_bar.update(1)
        
    progress_bar.set_postfix({"Status": "Processed"})
    df = pd.DataFrame(hist_data)  

    #Switch to real-time streaming
    while True:
        #Enter at the entry_id last found, i.e. the latest histroical id
        new_messages = await redis_client.xread({stream_key: last_id}, block=0)
        for stream, entries in new_messages:
            for entry_id, data in entries:
                data['close'] = float(data['close'])
                data['timestamp'] = datetime.fromisoformat(data['timestamp'])
                
                df.loc[len(df)] = data
                last_id = entry_id  # Update last processed ID
                
                progress_bar.update(1)
                progress_bar.set_postfix({"Status": "Streaming"})   
                progress_bar.colour = "#0cfade"
                
                await base_rsi(df, ticker)
                
        await asyncio.sleep(0)            
       

async def consume_ohlc_data_for_multiple_tickers(tickers, last_n=15):
    """Consume OHLC data for a list of tickers concurrently."""
    tasks = []
    
    for ticker in tickers:
        tasks.append(consume_ohlc_data(ticker, last_n))
    
    # Run all tickers concurrently
    await asyncio.gather(*tasks)

In [42]:
async def base_rsi(df, ticker):       
    OVERBOUGHT_THRESH = 70
    OVERSOLD_THRESH = 30
    CLOSE_POSITION_THRESH = 50

    # Need at least 14 bars to calculate RSI_14
    if len(df)<15:
        print('HOLD')
        return
        
    # Calculate Latest RSI
    rsi_value = rsi(df.close.iloc[-15:]).iloc[-1]
    #print(f"{ticker} RSI: {rsi_value}")
    # Trading Logic
    if rsi_value <= OVERSOLD_THRESH:
        # Buy when RSI is below the oversold threshold (enter long position)
        #print('buy', ticker)
        await place_market_order(ticker,'buy')
    
    elif rsi_value >= OVERBOUGHT_THRESH:
        # Sell when RSI is above the overbought threshold (enter long position)
        try:
            await place_market_order(ticker, 'sell')
            #print('sell', ticker)
        except:
            pass

    #await update_plot(ticker, rsi_value)

In [43]:
import plotly.graph_objects as go
import plotly.express as px
from collections import defaultdict
from plotly.subplots import make_subplots


rsi_values = defaultdict(list)
# Initialize an empty figure using FigureWidget
fig = go.FigureWidget(make_subplots(rows=1, cols=1))
fig.add_trace(go.Scatter(x=[], y=[], mode='lines', name="Placeholder"))  # Placeholder to avoid empty plot

# Configure layout
fig.update_layout(
    title="RSI for Multiple Tickers",
    xaxis_title="Time",
    yaxis_title="RSI Value",
    showlegend=True
)

# Display the initial figure (only needed once in Jupyter)
display(fig)

async def update_plot(ticker, rsi_value):
    """
    Dynamically updates the Plotly Express figure with new RSI values.
    """
    rsi_values[ticker].append(rsi_value)

    # Find existing trace for ticker, else create a new one
    for trace in fig.data:
        if trace.name == ticker:
            trace.x = list(range(len(rsi_values[ticker])))
            trace.y = rsi_values[ticker]
            break
    else:
        # Add a new trace if ticker doesn't exist
        fig.add_trace(go.Scatter(
            x=list(range(len(rsi_values[ticker]))),
            y=rsi_values[ticker],
            mode='lines',
            name=ticker
        ))
    
    # Update layout for clarity
    fig.update_layout(title="RSI for Multiple Tickers")


FigureWidget({
    'data': [{'mode': 'lines',
              'name': 'Placeholder',
              'type': 'scatter',
              'uid': '226d4b10-664c-4125-ae94-e8d721da9da7',
              'x': [],
              'y': []}],
    'layout': {'showlegend': True,
               'template': '...',
               'title': {'text': 'RSI for Multiple Tickers'},
               'xaxis': {'anchor': 'y', 'domain': [0.0, 1.0], 'title': {'text': 'Time'}},
               'yaxis': {'anchor': 'x', 'domain': [0.0, 1.0], 'title': {'text': 'RSI Value'}}}
})

In [44]:
await consume_ohlc_data_for_multiple_tickers(tickers)

0 1: | 

0 2: | 

0 3: | 

CancelledError: 